# Decode the Bioreactor — 05 Noise floor
**Question**: how much of the remaining error (~10 % MAPE) is measurement noise that no model can predict?

Idea: each training batch's own curve is smooth; the scatter of the measurements around it is noise. We estimate that scatter per state in three ways, on the log scale (log errors ≈ relative errors, like MAPE):

| method | what it does | bias |
|---|---|---|
| **A. neighbours** | compare each point with the average of its two neighbours (2nd difference); for white noise var = 1.5·σ² | **too high** where the curve bends (peak, feed jumps) |
| **B. leave-one-out spline** | fit a smoothing spline to the batch without point k, predict k | **too high** (includes interpolation error) |
| **C. in-sample spline** | smoothing spline through all points, smoothness chosen by GCV | **too low** (the spline also follows part of the noise) |

So the noise floor lies roughly **between C and min(A, B)**. Converted to MAPE with E|ε| = σ·√(2/π) for normal noise.

The gap between your model's CV MAPE and this floor is what a better model could still gain. *Caveat*: a model that predicts from day 0 + feeds also cannot know batch-specific wiggles that are real (not noise), so the achievable error is somewhat above the floor.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import make_smoothing_spline

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from bioreactor.data import load, STATES

df, tmpl, train_ids, test_ids = load(ROOT / "data")
s = df[df.batch_id.isin(train_ids) & df.vcd.notna()].sort_values(["batch_id", "elapsed_time"])
SQRT_2_PI = np.sqrt(2 / np.pi)       # E|eps| / sigma for normal noise
print(len(train_ids), "batches,", len(s), "samples")

## A. Neighbour (2nd-difference) estimate
e_k = log y_k − (log y_{k−1} + log y_{k+1}) / 2. For pure noise, var(e) = 1.5 σ². A robust version (MAD) limits the effect of single outliers.

In [ ]:
rows = []
for b, g in s.groupby("batch_id"):
    t = g.elapsed_time.to_numpy()
    for v in STATES:
        ly = np.log(g[v].to_numpy())
        # unequal spacing: linear interpolation between the neighbours instead of the plain mean
        w = (t[1:-1] - t[:-2]) / (t[2:] - t[:-2])
        e = ly[1:-1] - ((1 - w) * ly[:-2] + w * ly[2:])
        var_fac = 1 + (1 - w) ** 2 + w ** 2              # = 1.5 for equal spacing
        rows += [dict(batch_id=b, state=v, day=int(round(tk)), e_std=ek / np.sqrt(fk))
                 for tk, ek, fk in zip(t[1:-1], e, var_fac)]
A = pd.DataFrame(rows)

def sigma_mad(x):
    return 1.4826 * np.median(np.abs(x - np.median(x)))

est_A = A.groupby("state").e_std.agg(sigma_sd="std", sigma_mad=sigma_mad)
est_A["MAPE_floor_A_%"] = 100 * SQRT_2_PI * est_A.sigma_sd
est_A["MAPE_floor_A_robust_%"] = 100 * SQRT_2_PI * est_A.sigma_mad
est_A.loc[STATES].round(3)

## B. Leave-one-out and C. in-sample smoothing spline
`make_smoothing_spline` picks the smoothness by generalised cross-validation (GCV) per batch and state. Endpoints (day 0 and the last day) are skipped in B, because there the spline would extrapolate.

In [ ]:
rows_B, rows_C = [], []
for b, g in s.groupby("batch_id"):
    t = g.elapsed_time.to_numpy()
    for v in STATES:
        ly = np.log(g[v].to_numpy())
        # C: in-sample
        sp = make_smoothing_spline(t, ly)
        for tk, r in zip(t, ly - sp(t)):
            rows_C.append(dict(batch_id=b, state=v, day=int(round(tk)), err=r))
        # B: leave one out (interior points)
        for k in range(1, len(t) - 1):
            m = np.ones(len(t), bool); m[k] = False
            sp_k = make_smoothing_spline(t[m], ly[m])
            rows_B.append(dict(batch_id=b, state=v, day=int(round(t[k])), err=ly[k] - sp_k(t[k])))
B, C = pd.DataFrame(rows_B), pd.DataFrame(rows_C)

def ape(err):   # log error -> absolute percentage error of exp(err)
    return 100 * np.abs(np.expm1(err))

floor = pd.DataFrame({
    "C_in_sample_spline (low)": C.assign(a=ape(C.err)).groupby("state").a.mean(),
    "A_neighbours": est_A["MAPE_floor_A_%"],
    "A_neighbours_robust": est_A["MAPE_floor_A_robust_%"],
    "B_leave_one_out (high)": B.assign(a=ape(B.err)).groupby("state").a.mean(),
}).loc[STATES]
floor.loc["mean (= challenge score)"] = floor.mean()
floor.round(2)

## Compare with your models
Fill in the CV MAPE % per state of your best model and of the baseline (from notebooks 03 / 04).

In [ ]:
MY_CV = {   # <- fill in from the CV tables
    "per-day median": {"vcd": np.nan, "tcd": np.nan, "substrate": np.nan, "byproduct": np.nan},
    "best model":     {"vcd": np.nan, "tcd": np.nan, "substrate": np.nan, "byproduct": np.nan},
}
cmp = floor.loc[STATES, ["C_in_sample_spline (low)", "B_leave_one_out (high)"]].copy()
for name, d in MY_CV.items():
    cmp[name] = pd.Series(d)
cmp.loc["mean (= challenge score)"] = cmp.mean()
cmp.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(STATES))
lo = floor.loc[STATES, "C_in_sample_spline (low)"]; hi = floor.loc[STATES, "B_leave_one_out (high)"]
ax.bar(x, hi - lo, bottom=lo, width=0.5, color="0.8", label="noise floor range (C .. B)")
for i, (name, d) in enumerate(MY_CV.items()):
    vals = pd.Series(d).loc[STATES]
    if vals.notna().any():
        ax.plot(x + (i - 0.5) * 0.15, vals, "o", ms=8, label=name)
ax.set_xticks(x); ax.set_xticklabels(STATES); ax.set_ylabel("MAPE %"); ax.legend()
ax.set_title("model error vs. measurement-noise floor"); plt.tight_layout(); plt.show()

## Noise by day
Is the noise constant over the run, or larger at certain days (e.g. late cell counts, low early substrate)? Where model error ≫ noise on a day, that day is where improvement is possible.

In [ ]:
by_day = pd.concat({
    "C_low": C.assign(a=ape(C.err)).groupby(["state", "day"]).a.mean(),
    "B_high": B.assign(a=ape(B.err)).groupby(["state", "day"]).a.mean(),
}, axis=1).reset_index()
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
for a, v in zip(ax, STATES):
    d = by_day[by_day.state == v]
    a.fill_between(d.day, d.C_low, d.B_high.fillna(d.C_low), color="0.8", label="noise range")  # B has no endpoints
    a.plot(d.day, d.C_low, "k-", lw=1); a.plot(d.day, d.B_high, "k--", lw=1)
    a.set_title(v); a.set_xlabel("day"); a.set_ylabel("|% error|")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

## Examples: smoothing spline vs. data
Visual check that the spline is a sensible "true curve" (not too wiggly, not too stiff).

In [ ]:
EXAMPLES = train_ids[:3]
fig, ax = plt.subplots(len(EXAMPLES), 4, figsize=(16, 2.8 * len(EXAMPLES)))
for r, b in enumerate(EXAMPLES):
    g = s[s.batch_id == b]; t = g.elapsed_time.to_numpy(); tt = np.linspace(t.min(), t.max(), 200)
    for c, v in enumerate(STATES):
        sp = make_smoothing_spline(t, np.log(g[v].to_numpy()))
        ax[r, c].plot(t, g[v], "ko", ms=4); ax[r, c].plot(tt, np.exp(sp(tt)), "tab:blue")
        ax[r, c].set_title(f"{b}: {v}", fontsize=9)
plt.tight_layout(); plt.show()

## How to read the result
- **Model ≈ upper end of the range**: you are close to what the data allow; further modelling gains will be small. Focus on robustness, ensembling, and the story for the talk.
- **Model clearly above the range** (e.g. 10 % vs. a floor of 4–6 %): there is real, predictable batch-to-batch signal left. Then step 2 (feeds as measurements) and step 3 (whole-curve prediction) are worth the effort.
- Check per state and per day: the gain is usually concentrated in one state or phase.